<div align="center">

<a href="https://colab.research.google.com/github/utkukose/rd-project-management-NB-lecture/blob/main/weeks/week-10/NB10_risk_monte_carlo.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 10: Risk Management and Monte Carlo Simulation

**R&D and Project Management in Computer Science (11117BLG002)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/rd-project-management-NB-lecture/weeks/week-10/lab.html>.

## Overview

Research projects are uncertain by definition, so their plans must manage risk explicitly. This week follows the risk management process of ISO 31000, compares qualitative risk matrices with quantitative analysis and uses Monte Carlo simulation to estimate schedule risk [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to identify, analyse and treat project risks with a risk register, to compute risk exposure, to explain the limitations of qualitative risk matrices, and to estimate the distribution of project duration with Monte Carlo simulation, including merge bias.

## The risk management process

ISO 31000 defines risk as the effect of uncertainty on objectives and describes risk management through principles, a framework and a process [1]. The process identifies risks, analyses their likelihood and consequences, evaluates them against criteria, treats them and monitors them, with communication and consultation throughout. Treatment options include avoiding the risk, reducing its likelihood or consequences, sharing it and accepting it. In a proposal, the risk section shows reviewers that the team has thought about what could go wrong and has credible alternatives, which TÜBİTAK's form asks for as a plan B [4].

## Risk registers and exposure

Boehm introduced risk management to software engineering and defined risk exposure as the probability of an unsatisfactory outcome multiplied by the loss it would cause [2]. A risk register lists each risk with its probability, impact, exposure, owner, mitigation and trigger. Boehm's list of frequent software risks, which includes personnel shortfalls and unrealistic schedules and budgets, remains familiar in research projects, together with risks specific to research: a hypothesis that turns out false, data that cannot be obtained, and approvals that take longer than planned.

## The limits of risk matrices

Probability-impact matrices are popular because they are simple. Hubbard argued that many common risk scoring methods, including ordinal scales and matrices, can mislead: They compress different risks into the same cell, rank risks inconsistently and give an impression of precision that the underlying judgements do not have [5]. He recommended quantitative methods based on calibrated probability estimates. The lab uses a matrix for communication and a simulation for decisions, which reflects this advice.

## Monte Carlo schedule risk

Monte Carlo simulation replaces single durations with probability distributions, samples every task many times and computes the project duration for each sample [3]. The result is a distribution from which percentiles such as P50 and P80 can be read, the durations that the project will meet with 50 and 80 percent probability. Figure 10.1 applies the method to the network of Week 9 with right-skewed durations. Two effects make the simulated durations longer than the deterministic plan of 20 weeks. Skewed tasks are more likely to overrun than to finish early, and where paths merge, the later of the two paths determines the start of the next task, so the expected merge time exceeds the time of either path. This merge bias is invisible to PERT.

> **Pause and reflect.** Would you commit to the P50 or the P80 duration in a proposal? What would the funder and your team each prefer?

![Figure 10.1](https://raw.githubusercontent.com/utkukose/rd-project-management-NB-lecture/main/weeks/week-10/figures/w10_fig1.png)

*Figure 10.1. Distribution of the duration of the example project in 20,000 Monte Carlo runs with triangular task durations from 0.9 to 1.6 times the planned value [3].*

# Hands-on lab

The notebook simulates the duration of the Week 9 project with triangular task durations, reads percentiles from the distribution and identifies the tasks that drive the uncertainty [3].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Monte Carlo simulation

In [ ]:
NET = [("A", 3, []), ("B", 2, []), ("C", 5, ["B"]), ("D", 4, ["A"]), ("E", 6, ["C", "D"]), ("F", 2, ["D"]), ("G", 4, ["E", "F"]), ("H", 3, ["G"]), ("I", 2, ["G"])]

def simulate(n=20000, low=0.9, high=1.6, seed=SEED):
    rng = np.random.default_rng(seed)
    dur = {t: rng.triangular(low * m, m, high * m, n) for t, m, _ in NET}
    ef = {}
    for t, _, pred in NET:
        start = np.max([ef[q] for q in pred], axis=0) if pred else np.zeros(n)
        ef[t] = start + dur[t]
    return np.max(np.vstack(list(ef.values())), axis=0), dur

total, durations = simulate()
plt.hist(total, bins=60); plt.axvline(20, color="k"); plt.xlabel("weeks"); plt.title("Simulated project duration"); plt.show()

### Your turn, exercise 1

Compute the P80 duration, the 80th percentile of `total`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _p80_reference():
    return float(np.percentile(total, 80))

In [ ]:
p80 = None  # your computation
check("Exercise 1", p80, _p80_reference())

### Your turn, exercise 2

Compute the probability of finishing within 22 weeks and compare it with the PERT estimate from Week 9. Then compute the correlation of each task's duration with the total, and name the two tasks that drive the uncertainty most.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _p22_reference():
    return float(np.mean(total <= 22))

In [ ]:
p_within_22 = None  # your computation
check("Exercise 2", p_within_22, _p22_reference())
corr = {t: float(np.corrcoef(d, total)[0, 1]) for t, d in durations.items()}
print({t: round(c, 2) for t, c in sorted(corr.items(), key=lambda kv: -kv[1])})

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/rd-project-management-NB-lecture/weeks/week-10/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which risk in your register has the highest exposure, and is your mitigation proportionate to it?
2. Why might a team prefer to report P50 in a proposal while a funder would prefer P80?
3. Which task drives the uncertainty of your schedule, and can it be started earlier or split?

## Weekly task and submission

Build a risk register of at least eight risks for your project with probability, impact, exposure, owner, mitigation and plan B, and run a Monte Carlo analysis of your schedule. Report P50 and P80 and justify the duration you would commit to. Attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Quantitative risk analysis in practice.** Review how organisations apply ISO 31000 and Monte Carlo analysis to projects and summarise the evidence on the limitations of qualitative risk matrices [1, 3, 5].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] ISO (2018). *ISO 31000:2018 Risk management: Guidelines*. International Organization for Standardization.

[2] Boehm, B. W. (1991). Software risk management: Principles and practices. *IEEE Software*, 8(1), 32-41. <https://doi.org/10.1109/52.62930>

[3] Vose, D. (2008). *Risk Analysis: A Quantitative Guide* (3rd ed.). Wiley.

[4] TÜBİTAK (2026). *1001 Bilimsel ve Teknolojik Araştırma Projelerini Destekleme Programı: Proje başvuru formu*. <https://tubitak.gov.tr/sites/default/files/20689/1001_basvuru_formu.doc>

[5] Hubbard, D. W. (2009). *The Failure of Risk Management: Why It's Broken and How to Fix It*. Wiley.